In [ ]:
# %pip install PyYAML
# %pip install dynamic-yaml


In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()
DATA_FOLDER_PATH = os.getenv("DATA_FOLDER_PATH")
CONSTANTS_CONFIG_FOLDER_PATH = os.getenv("CONSTANTS_CONFIG_FOLDER_PATH")
PROCESSING_CONFIG_FOLDER_PATH = os.getenv("PROCESSING_CONFIG_FOLDER_PATH")
VALIDATION_CONFIG_FOLDER_PATH = os.getenv("VALIDATION_CONFIG_FOLDER_PATH")
DATA_FOLDER_PATH, CONSTANTS_CONFIG_FOLDER_PATH

In [ ]:
import yaml
import pandas as pd

with open("/home/user/datavalidation/configs/file_info.yaml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    df_file_info = pd.DataFrame(data)
    # print(type(data), json.dumps(data, indent=2))
df_file_info = df_file_info.explode("sheets", ignore_index=True)
df_file_info["file"] = DATA_FOLDER_PATH + "/" + df_file_info["file"]
df_file_info["config_file"] = CONSTANTS_CONFIG_FOLDER_PATH + "/" + df_file_info["config_file"]
df_file_info["processing_config_file"] = PROCESSING_CONFIG_FOLDER_PATH + "/" + df_file_info["processing_config_file"]
df_file_info["validation_config_file"] = VALIDATION_CONFIG_FOLDER_PATH + "/" + df_file_info["validation_config_file"]
df_file_info

In [ ]:
common_kwargs = dict()
file_name_mapping = df_file_info[["file_name", "file_name_mapping"]].dropna().drop_duplicates().to_dict(orient="records")
# file_name_mapping = {file_name_mapping.loc[index, "file_name_mapping"]: file_name_mapping.loc[index, "file_name"] for index, item in df_file_info.iterrows()}
file_name_mapping

for item in file_name_mapping:
    common_kwargs[item["file_name_mapping"]] = item["file_name"]

# common_kwargs = {**common_kwargs, **file_name_mapping}
common_kwargs


In [ ]:
pd.read_excel(df_file_info.loc[0, "file"], sheet_name=df_file_info.loc[0, "sheets"])

In [ ]:
import pandas as pd
pd.DataFrame({"A": [True, False, 1, "1"]}).map(lambda x: {
    True: "true",
    False: "false"
}.get(x, x))

In [ ]:
import yaml
import json
import pandas as pd

with open("/home/user/datavalidation/configs/validation/master_data_setup_validation_config.yaml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    df = pd.DataFrame(data)
    # print(type(data), json.dumps(data, indent=2))
df

In [ ]:
df_files = pd.json_normalize(df["files"]).explode("columns", ignore_index=True)
df_columns = pd.json_normalize(df_files["columns"]).explode("rules", ignore_index=True)
df_files = pd.concat([df_files,df_columns], axis=1)
df_rules = pd.json_normalize(df_files["rules"])
df_files = pd.concat([df_files,df_rules], axis=1)

df_files

In [ ]:
df_files["file_path"] = df_files["file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files.loc[0, "file_path"]
df_files["report_file_path"] = df_files["report_file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files["ref_file"] = df_files["ref_file"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])

In [ ]:
df_files_copy = df_files.copy()
df_files_copy.drop(["columns", "rules"], axis=1, inplace=True)
df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]] = df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]].ffill()
df_files_copy